# Bangalore Climate Analytics & Forecasting
## Multi-Variable Climate Analysis | Anomaly Detection | CES | Forecasting
**Location:** Bangalore, India | **Period:** 1951-2024

---
This notebook implements a complete climate analytics pipeline:
1. Data loading and preprocessing
2. Exploratory data analysis
3. Climatology baseline
4. Anomaly detection (Z-score)
5. Climate Extreme Score (CES)
6. Seasonal and decade-wise analysis
7. Climate shift analysis
8. SARIMA forecasting
9. Future climate extreme projections
10. Automated insights


## Step 1 - Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, joblib
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
for d in ['saved_models', 'outputs', 'plots']:
    os.makedirs(d, exist_ok=True)
print('Libraries imported.')


Libraries imported.


## Step 2 - Load Dataset

In [3]:
DATA_PATH = 'data.csv'
df_raw = pd.read_csv(DATA_PATH)
print('Shape:', df_raw.shape)
print('Columns:', list(df_raw.columns))
print(df_raw.head())
print('Missing values:')
print(df_raw.isnull().sum())


Shape: (26835, 5)
Columns: ['Date', 'Rain', 'Temp Max', 'Temp Min', 'Humidity']
         Date Rain   Temp Max  Temp Min  Humidity
0  1951-01-01    0  26.969999     15.65     77.36
1  1951-01-02    0  26.180000     14.69     77.02
2  1951-01-03    0  26.309999     14.95     77.28
3  1951-01-04    0  26.730000     14.84     76.22
4  1951-01-05    0  26.930000     14.53     75.20
Missing values:
Date         0
Rain         8
Temp Max     7
Temp Min    10
Humidity    10
dtype: int64


## Step 3 - Data Preprocessing
- Rename columns, parse dates
- Interpolate missing values
- Compute Temp_mean = (Temp_Max + Temp_Min) / 2
- Aggregate daily to monthly


In [4]:
df = df_raw.copy()
df.columns = ['Date', 'Rainfall', 'Temp_Max', 'Temp_Min', 'Humidity']
df['Date'] = pd.to_datetime(df['Date'])
df = df.drop_duplicates(subset='Date').reset_index(drop=True)

for col in ['Rainfall', 'Temp_Max', 'Temp_Min', 'Humidity']:
    df[col] = df[col].interpolate(method='linear').ffill().bfill()

df['Temp_mean'] = (df['Temp_Max'] + df['Temp_Min']) / 2
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

def get_season(m):
    if m in [12,1,2]:    return 'Winter'
    elif m in [3,4,5]:   return 'Summer'
    elif m in [6,7,8,9]: return 'Monsoon'
    else:                return 'Post-Monsoon'

df['Season'] = df['Month'].apply(get_season)
df['Decade'] = (df['Year'] // 10) * 10
print('Daily dataset shape:', df.shape)
print(df.head(3))


Daily dataset shape: (26835, 10)
        Date Rainfall   Temp_Max  Temp_Min  Humidity  Temp_mean  Year  Month  \
0 1951-01-01        0  26.969999     15.65     77.36  21.309999  1951      1   
1 1951-01-02        0  26.180000     14.69     77.02  20.435000  1951      1   
2 1951-01-03        0  26.309999     14.95     77.28  20.630000  1951      1   

   Season  Decade  
0  Winter    1950  
1  Winter    1950  
2  Winter    1950  


In [5]:
monthly = df.groupby(['Year', 'Month']).agg(
    Rainfall=('Rainfall', 'sum'),
    Temp_Max=('Temp_Max', 'mean'),
    Temp_Min=('Temp_Min', 'mean'),
    Temp_mean=('Temp_mean', 'mean'),
    Humidity=('Humidity', 'mean')
).reset_index()

monthly['Date'] = pd.to_datetime(
    monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2) + '-01'
)
monthly = monthly.sort_values('Date').reset_index(drop=True)
monthly['Season'] = monthly['Month'].apply(get_season)
monthly['Decade'] = (monthly['Year'] // 10) * 10

print('Monthly shape:', monthly.shape)
print(monthly.head())
monthly.to_csv('outputs/monthly_processed.csv', index=False)
print('Saved: outputs/monthly_processed.csv')


Monthly shape: (882, 10)
   Year  Month                                           Rainfall   Temp_Max  \
0  1951      1                    0000000000000000000000000000000  27.460645   
1  1951      2                       0000000000000000000000000000  30.429286   
2  1951      3  0000000000000000000000026.18957713.45926904700...  33.200645   
3  1951      4  5.6454334261.333566904000000006.2233123788.233...  32.598666   
4  1951      5  0.4468939012.467976332000000001.633867864.3253...  33.304516   

    Temp_Min  Temp_mean   Humidity       Date  Season  Decade  
0  15.657742  21.559194  76.394194 1951-01-01  Winter    1950  
1  16.232857  23.331071  71.607143 1951-02-01  Winter    1950  
2  19.285806  26.243226  72.170323 1951-03-01  Summer    1950  
3  21.725666  27.162166  78.254000 1951-04-01  Summer    1950  
4  22.026774  27.665645  77.444516 1951-05-01  Summer    1950  
Saved: outputs/monthly_processed.csv


## Step 4 - Exploratory Data Analysis

In [8]:
def plot_trend(df, col, ylabel, title, color='steelblue', filename=None):
    df = df.copy()
    df[col] = pd.to_numeric(df[col], errors='coerce')
    yearly = df.groupby('Year')[col].mean()
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(yearly.index, yearly.values, color=color, alpha=0.5, label='Annual Mean')
    roll = yearly.rolling(5, center=True).mean()
    ax.plot(roll.index, roll.values, color='red', linewidth=2, label='5-yr Rolling Avg')
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if filename:
        plt.savefig(f'plots/{filename}', dpi=100)
    plt.show()

plot_trend(monthly, 'Rainfall', 'Rainfall (mm)', 'Bangalore Monthly Rainfall 1951-2024',
           color='royalblue', filename='01_rainfall_trend.png')
plot_trend(monthly, 'Temp_mean', 'Temperature (C)', 'Bangalore Mean Temperature 1951-2024',
           color='tomato', filename='02_temperature_trend.png')
plot_trend(monthly, 'Humidity', 'Humidity (%)', 'Bangalore Humidity 1951-2024',
           color='seagreen', filename='03_humidity_trend.png')

In [10]:
for col in ['Rainfall', 'Temp_mean', 'Humidity']:
    monthly[col] = pd.to_numeric(monthly[col], errors='coerce')

In [11]:
# Correlation heatmap
cols = ['Rainfall', 'Temp_mean', 'Humidity']
corr = monthly[cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha='right')
ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('plots/04_correlation_heatmap.png', dpi=100)
plt.show()


In [12]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (col, color) in zip(axes, [('Rainfall','royalblue'),('Temp_mean','tomato'),('Humidity','seagreen')]):
    ax.hist(monthly[col], bins=40, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(monthly[col].mean(), color='black', linestyle='--', label='Mean')
    ax.set_title(f'{col} Distribution')
    ax.legend()
fig.suptitle('Climate Variable Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('plots/05_distributions.png', dpi=100)
plt.show()


## Step 5 - Climatology Baseline
Monthly mean and std for each variable, used as reference for anomaly detection.

In [13]:
baseline = monthly.groupby('Month').agg(
    Rain_mean=('Rainfall', 'mean'),
    Rain_std=('Rainfall', 'std'),
    Temp_mean_mean=('Temp_mean', 'mean'),
    Temp_mean_std=('Temp_mean', 'std'),
    Hum_mean=('Humidity', 'mean'),
    Hum_std=('Humidity', 'std')
).reset_index()

print('Monthly Climatology Baseline:')
print(baseline.round(2).to_string(index=False))
baseline.to_csv('outputs/climatology_baseline.csv', index=False)


Monthly Climatology Baseline:
 Month  Rain_mean  Rain_std  Temp_mean_mean  Temp_mean_std  Hum_mean  Hum_std
     1       0.39      2.04           21.87           0.65     74.82     2.23
     2       0.22      1.22           24.05           0.66     72.44     2.21
     3       0.35      1.35           26.68           0.70     72.19     1.84
     4       3.88      6.23           28.43           0.70     74.41     1.46
     5       0.34      0.47           27.99           0.79     76.47     1.97
     6      15.33       NaN           25.74           0.62     81.20     1.43
     7        NaN       NaN           24.72           0.55     83.14     1.38
     8        NaN       NaN           24.46           0.45     83.31     1.17
     9        NaN       NaN           24.66           0.47     82.06     1.47
    10        NaN       NaN           24.28           0.47     82.14     1.63
    11       0.61      1.22           22.77           0.60     81.12     2.28
    12       1.42      2.83       

## Step 6 - Anomaly Detection
Z = (X - Mean) / StdDev

Labels: |Z| >= 2 -> Strong | 1.5 <= |Z| < 2 -> Moderate | else -> Normal


In [14]:
mdf = monthly.merge(baseline, on='Month', how='left')

# Raw anomalies
mdf['Rain_anomaly'] = mdf['Rainfall'] - mdf['Rain_mean']
mdf['Temp_anomaly'] = mdf['Temp_mean'] - mdf['Temp_mean_mean']
mdf['Hum_anomaly']  = mdf['Humidity']  - mdf['Hum_mean']

# Z-scores
mdf['Z_rain'] = mdf['Rain_anomaly'] / mdf['Rain_std'].replace(0, np.nan).fillna(1)
mdf['Z_temp'] = mdf['Temp_anomaly'] / mdf['Temp_mean_std'].replace(0, np.nan).fillna(1)
mdf['Z_hum']  = mdf['Hum_anomaly']  / mdf['Hum_std'].replace(0, np.nan).fillna(1)

def label_z(z):
    a = abs(z)
    return 'Strong' if a >= 2 else ('Moderate' if a >= 1.5 else 'Normal')

mdf['Rain_label'] = mdf['Z_rain'].apply(label_z)
mdf['Temp_label'] = mdf['Z_temp'].apply(label_z)
mdf['Hum_label']  = mdf['Z_hum'].apply(label_z)

# CES
mdf['CES'] = mdf['Z_rain'] + mdf['Z_temp'] + mdf['Z_hum']

def label_ces(c):
    a = abs(c)
    return 'Extreme' if a >= 3 else ('Moderate' if a >= 1 else 'Normal')

mdf['CES_label'] = mdf['CES'].apply(label_ces)

print('Anomaly counts:')
print('Rainfall:', mdf['Rain_label'].value_counts().to_dict())
print('Temperature:', mdf['Temp_label'].value_counts().to_dict())
print('Humidity:', mdf['Hum_label'].value_counts().to_dict())
print('CES:', mdf['CES_label'].value_counts().to_dict())


Anomaly counts:
Rainfall: {'Normal': 870, 'Strong': 8, 'Moderate': 4}
Temperature: {'Normal': 768, 'Moderate': 73, 'Strong': 41}
Humidity: {'Normal': 777, 'Moderate': 63, 'Strong': 42}
CES: {'Normal': 760, 'Moderate': 97, 'Extreme': 25}


In [15]:
# Anomaly Z-score time series
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
z_pairs = [('Z_rain','Rainfall Z-score','royalblue'),
           ('Z_temp','Temperature Z-score','tomato'),
           ('Z_hum','Humidity Z-score','seagreen')]
for ax, (col, label, color) in zip(axes, z_pairs):
    ax.bar(mdf['Date'], mdf[col], color=color, alpha=0.5, width=20)
    ax.axhline(2,   color='red',    linestyle='--', linewidth=0.8, label='+/-2sigma')
    ax.axhline(-2,  color='red',    linestyle='--', linewidth=0.8)
    ax.axhline(1.5, color='orange', linestyle=':',  linewidth=0.8, label='+/-1.5sigma')
    ax.axhline(-1.5,color='orange', linestyle=':',  linewidth=0.8)
    ax.set_ylabel(label)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2)
axes[0].set_title('Climate Anomalies (Z-scores) 1951-2024')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('plots/06_anomaly_zscores.png', dpi=100)
plt.show()


## Step 7 - Climate Extreme Score (CES)
CES = Z_temp + Z_rain + Z_humidity

|CES| < 1 -> Normal | 1 <= |CES| < 3 -> Moderate | |CES| >= 3 -> Extreme


In [16]:
fig, ax = plt.subplots(figsize=(13, 4))
ces_colors = mdf['CES_label'].map({'Normal':'steelblue','Moderate':'orange','Extreme':'red'})
ax.bar(mdf['Date'], mdf['CES'], color=ces_colors, width=20, alpha=0.7)
ax.axhline(3,  color='red',    linestyle='--', linewidth=1, label='Extreme (+/-3)')
ax.axhline(-3, color='red',    linestyle='--', linewidth=1)
ax.axhline(1,  color='orange', linestyle=':',  linewidth=1, label='Moderate (+/-1)')
ax.axhline(-1, color='orange', linestyle=':',  linewidth=1)
ax.set_title('Climate Extreme Score (CES) - Bangalore 1951-2024')
ax.set_xlabel('Date')
ax.set_ylabel('CES')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('plots/07_ces_timeseries.png', dpi=100)
plt.show()


In [17]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mdf['CES'], bins=60, color='purple', alpha=0.7, edgecolor='white')
ax.axvline(3,  color='red',    linestyle='--', label='Extreme')
ax.axvline(-3, color='red',    linestyle='--')
ax.axvline(1,  color='orange', linestyle=':',  label='Moderate')
ax.axvline(-1, color='orange', linestyle=':')
ax.set_title('CES Distribution')
ax.set_xlabel('CES')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('plots/08_ces_distribution.png', dpi=100)
plt.show()


## Step 8 - Anomaly Evolution Analysis
Year-wise and decade-wise anomaly count trends.

In [18]:
mdf['any_anom'] = ((mdf['Rain_label'] != 'Normal') |
                   (mdf['Temp_label'] != 'Normal') |
                   (mdf['Hum_label']  != 'Normal')).astype(int)
yearly_anom = mdf.groupby('Year')['any_anom'].sum().reset_index()

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(yearly_anom['Year'], yearly_anom['any_anom'], color='coral', alpha=0.8)
roll = yearly_anom.set_index('Year')['any_anom'].rolling(5, center=True).mean()
ax.plot(roll.index, roll.values, color='darkred', linewidth=2, label='5-yr trend')
ax.set_title('Annual Anomaly Count Evolution 1951-2024')
ax.set_xlabel('Year')
ax.set_ylabel('Anomalous Months')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('plots/09_anomaly_evolution.png', dpi=100)
plt.show()


In [19]:
decade_anom = mdf.groupby('Decade')['any_anom'].sum().reset_index()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(decade_anom['Decade'].astype(str), decade_anom['any_anom'], color='coral', alpha=0.8)
ax.set_title('Decade-wise Anomaly Count')
ax.set_xlabel('Decade')
ax.set_ylabel('Count')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('plots/10_decade_anomaly.png', dpi=100)
plt.show()


## Step 9 - Seasonal Analysis
Winter | Summer | Monsoon | Post-Monsoon comparison.

In [20]:
season_order = ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']
season_df = mdf.groupby('Season').agg(
    Rainfall=('Rainfall', 'mean'),
    Temp_mean=('Temp_mean', 'mean'),
    Humidity=('Humidity', 'mean'),
    CES=('CES', 'mean')
).reindex(season_order)

print('Seasonal Climate Summary:')
print(season_df.round(2))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
spairs = [('Rainfall','royalblue'),('Temp_mean','tomato'),('Humidity','seagreen'),('CES','purple')]
for ax, (col, color) in zip(axes.flatten(), spairs):
    ax.bar(season_df.index, season_df[col], color=color, alpha=0.8)
    ax.set_title(f'Seasonal {col}')
    ax.grid(True, alpha=0.3, axis='y')
fig.suptitle('Seasonal Climate Analysis - Bangalore', fontsize=14)
plt.tight_layout()
plt.savefig('plots/11_seasonal_analysis.png', dpi=100)
plt.show()


Seasonal Climate Summary:
              Rainfall  Temp_mean  Humidity   CES
Season                                           
Winter            0.55      22.49     75.31 -0.32
Summer            1.04      27.70     74.36  0.08
Monsoon          15.33      24.90     82.42 -0.23
Post-Monsoon      0.61      23.52     81.63 -1.79


## Step 10 - Climate Shift Analysis
Compare 1951-1980 | 1981-2000 | 2001-2024

In [21]:
bins   = [1950, 1980, 2000, 2025]
labels = ['1951-1980', '1981-2000', '2001-2024']
mdf['Period'] = pd.cut(mdf['Year'], bins=bins, labels=labels, right=True)

period_df = mdf.groupby('Period', observed=True).agg(
    Rainfall=('Rainfall', 'mean'),
    Temp_mean=('Temp_mean', 'mean'),
    Humidity=('Humidity', 'mean'),
    CES=('CES', 'mean'),
    Anomaly_count=('any_anom', 'sum')
).reset_index()

print('Climate Shift Across Periods:')
print(period_df.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(15, 5))
ppairs = [('Rainfall','royalblue'),('Temp_mean','tomato'),('Humidity','seagreen'),('CES','purple')]
for ax, (col, color) in zip(axes, ppairs):
    ax.bar(period_df['Period'].astype(str), period_df[col], color=color, alpha=0.85)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3, axis='y')
fig.suptitle('Climate Shift Across Historical Periods - Bangalore', fontsize=13)
plt.tight_layout()
plt.savefig('plots/12_climate_shift.png', dpi=100)
plt.show()
period_df.to_csv('outputs/period_comparison.csv', index=False)


Climate Shift Across Periods:
   Period  Rainfall  Temp_mean  Humidity    CES  Anomaly_count
1951-1980     0.688     24.512    78.938 -0.408             83
1981-2000     0.684     24.834    77.949 -0.395             41
2001-2024     0.929     25.055    78.343  0.202             75


## Step 11 - Time-Series Forecasting (SARIMA)
Train SARIMA models for Rainfall, Temperature, Humidity.
- Train: 1951-2018 | Test: 2019-2024


In [22]:
def adf_test(series, name=''):
    result = adfuller(series.dropna())
    stat = result[1] < 0.05
    print(f'ADF [{name}]: p={result[1]:.4f} -> {"Stationary" if stat else "Non-Stationary"}')
    return stat

def evaluate(actual, predicted):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / (actual + 1e-8))) * 100
    r2   = r2_score(actual, predicted)
    return {'MAE': round(mae,3), 'RMSE': round(rmse,3), 'MAPE%': round(mape,3), 'R2': round(r2,3)}

TRAIN_CUTOFF = 2018
train_m = mdf[mdf['Year'] <= TRAIN_CUTOFF].set_index('Date')
test_m  = mdf[mdf['Year'] >  TRAIN_CUTOFF].set_index('Date')
print(f'Train: {train_m.index.min().date()} to {train_m.index.max().date()} ({len(train_m)} months)')
print(f'Test : {test_m.index.min().date()} to {test_m.index.max().date()} ({len(test_m)} months)')
targets = ['Rainfall', 'Temp_mean', 'Humidity']
for var in targets:
    adf_test(train_m[var], var)


Train: 1951-01-01 to 2018-12-01 (816 months)
Test : 2019-01-01 to 2024-06-01 (66 months)
ADF [Rainfall]: p=0.0000 -> Stationary
ADF [Temp_mean]: p=0.0023 -> Stationary
ADF [Humidity]: p=0.0091 -> Stationary


In [24]:
all_results = {}
all_metrics = {}

for var in targets:
    print(f'--- Training SARIMA for {var} ---')
    train_s = pd.to_numeric(train_m[var], errors='coerce').fillna(method='ffill').fillna(method='bfill')
    test_s  = pd.to_numeric(test_m[var],  errors='coerce').fillna(method='ffill').fillna(method='bfill')

    model = SARIMAX(train_s, order=(1,1,1), seasonal_order=(1,1,1,12),
                    enforce_stationarity=False, enforce_invertibility=False)
    fitted = model.fit(disp=False)

    fc_test = fitted.get_forecast(steps=len(test_s))
    pred    = fc_test.predicted_mean

    # Clean both arrays before evaluation
    actual_vals = pd.to_numeric(test_s.values[:len(pred)], errors='coerce')
    pred_vals   = pd.to_numeric(pred.values, errors='coerce')

    # Drop positions where either is NaN/inf
    mask = np.isfinite(actual_vals) & np.isfinite(pred_vals)
    actual_clean = actual_vals[mask]
    pred_clean   = pred_vals[mask]

    m = evaluate(actual_clean, pred_clean)
    all_metrics[var] = m
    print(f'  MAE={m["MAE"]}  RMSE={m["RMSE"]}  MAPE={m["MAPE%"]}%  R2={m["R2"]}')

    joblib.dump(fitted, f'saved_models/sarima_{var.lower()}.pkl')
    print(f'  Saved: saved_models/sarima_{var.lower()}.pkl')

    all_results[var] = {'model': fitted, 'train': train_s, 'test': test_s,
                        'pred_test': pd.Series(pred.values, index=test_s.index[:len(pred)])}

print('\nEvaluation Metrics:')
print(pd.DataFrame(all_metrics).T.to_string())
pd.DataFrame(all_metrics).T.reset_index().to_csv('outputs/model_evaluation.csv', index=False)

--- Training SARIMA for Rainfall ---
  MAE=1.054  RMSE=1.408  MAPE=9352412018.716%  R2=-1.653
  Saved: saved_models/sarima_rainfall.pkl
--- Training SARIMA for Temp_mean ---
  MAE=0.754  RMSE=0.918  MAPE=3.004%  R2=0.803
  Saved: saved_models/sarima_temp_mean.pkl
--- Training SARIMA for Humidity ---
  MAE=2.392  RMSE=2.882  MAPE=2.966%  R2=0.63
  Saved: saved_models/sarima_humidity.pkl

Evaluation Metrics:
             MAE   RMSE         MAPE%     R2
Rainfall   1.054  1.408  9.352412e+09 -1.653
Temp_mean  0.754  0.918  3.004000e+00  0.803
Humidity   2.392  2.882  2.966000e+00  0.630


In [25]:
colors_map = ['royalblue', 'tomato', 'seagreen']
fig, axes = plt.subplots(3, 1, figsize=(13, 12))
for ax, var, color in zip(axes, targets, colors_map):
    res = all_results[var]
    ax.plot(res['train'].index, res['train'].values, color=color, alpha=0.4, label='Train')
    ax.plot(res['test'].index,  res['test'].values,  color='green', label='Actual (Test)')
    ax.plot(res['pred_test'].index, res['pred_test'].values, color='red', linestyle='--', label='Predicted')
    ax.set_title(f'{var} - Actual vs Predicted')
    ax.set_ylabel(var)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
plt.suptitle('SARIMA Forecast - Actual vs Predicted (2019-2024)', fontsize=13)
plt.tight_layout()
plt.savefig('plots/13_actual_vs_predicted.png', dpi=100)
plt.show()


In [26]:
FUTURE_STEPS = 24
future_dict = {}
for var in targets:
    fc = all_results[var]['model'].get_forecast(steps=FUTURE_STEPS)
    future_dict[var] = fc.predicted_mean

future_df = pd.DataFrame(future_dict).reset_index()
future_df.columns = ['Date'] + targets
print('Future Forecast (24 months):')
print(future_df.round(2).to_string(index=False))
future_df.to_csv('outputs/future_forecast.csv', index=False)


Future Forecast (24 months):
      Date  Rainfall  Temp_mean  Humidity
2019-01-01     -0.78      22.76     74.07
2019-02-01     -0.78      24.84     71.64
2019-03-01     -0.53      27.38     71.49
2019-04-01      0.31      29.09     73.78
2019-05-01      0.43      28.74     75.75
2019-06-01      0.76      26.46     80.65
2019-07-01      0.84      25.41     82.40
2019-08-01      0.92      25.12     82.69
2019-09-01      0.97      25.24     81.39
2019-10-01      1.02      24.89     81.39
2019-11-01      1.10      23.29     80.32
2019-12-01      1.37      22.01     77.76
2020-01-01      0.36      22.42     73.90
2020-02-01      0.16      24.64     71.59
2020-03-01      0.25      27.27     71.46
2020-04-01      0.97      29.05     73.76
2020-05-01      0.97      28.65     75.73
2020-06-01      1.21      26.36     80.62
2020-07-01      1.22      25.37     82.38
2020-08-01      1.22      25.07     82.67
2020-09-01      1.23      25.30     81.37
2020-10-01      1.24      24.89     81.37
2020-

In [27]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
for ax, var, color in zip(axes, targets, colors_map):
    hist_tail = all_results[var]['train'].iloc[-36:]
    ax.plot(hist_tail.index, hist_tail.values, color=color, alpha=0.5, label='Historical (tail)')
    ax.plot(future_df['Date'], future_df[var], color='orange', marker='o', markersize=3, linestyle='-.', label='Future Forecast')
    ax.set_title(f'{var} - Future 24-Month Forecast')
    ax.set_ylabel(var)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
plt.suptitle('Bangalore Climate - Future Forecast', fontsize=13)
plt.tight_layout()
plt.savefig('plots/14_future_forecast.png', dpi=100)
plt.show()


## Step 11.7 - Future Climate Extreme Analysis
Compute CES for forecasted future values.

In [28]:
future_ces = future_df.copy()
for var, key in [('Rainfall','rain'), ('Temp_mean','temp'), ('Humidity','hum')]:
    mu  = mdf[var].mean()
    sig = mdf[var].std()
    future_ces[f'Z_{key}'] = (future_ces[var] - mu) / (sig + 1e-8)

future_ces['CES'] = future_ces['Z_rain'] + future_ces['Z_temp'] + future_ces['Z_hum']
future_ces['CES_label'] = future_ces['CES'].apply(label_ces)

print('Future CES Summary:')
print(future_ces[['Date','Z_rain','Z_temp','Z_hum','CES','CES_label']].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4))
bc = future_ces['CES_label'].map({'Normal':'steelblue','Moderate':'orange','Extreme':'red'})
ax.bar(future_ces['Date'], future_ces['CES'], color=bc, alpha=0.8)
ax.axhline(3,  color='red',    linestyle='--', label='Extreme')
ax.axhline(-3, color='red',    linestyle='--')
ax.axhline(1,  color='orange', linestyle=':',  label='Moderate')
ax.axhline(-1, color='orange', linestyle=':')
ax.set_title('Future CES - 24-Month Forecast')
ax.set_xlabel('Date')
ax.set_ylabel('CES')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('plots/15_future_ces.png', dpi=100)
plt.show()
future_ces.to_csv('outputs/future_ces.csv', index=False)


Future CES Summary:
      Date  Z_rain  Z_temp  Z_hum    CES CES_label
2019-01-01  -0.586  -0.925 -0.995 -2.506  Moderate
2019-02-01  -0.587   0.032 -1.542 -2.096  Moderate
2019-03-01  -0.491   1.198 -1.575 -0.868    Normal
2019-04-01  -0.169   1.982 -1.059  0.755    Normal
2019-05-01  -0.126   1.821 -0.614  1.081  Moderate
2019-06-01   0.000   0.774  0.489  1.263  Moderate
2019-07-01   0.033   0.294  0.885  1.212  Moderate
2019-08-01   0.061   0.159  0.950  1.169  Moderate
2019-09-01   0.083   0.216  0.657  0.956    Normal
2019-10-01   0.102   0.051  0.656  0.810    Normal
2019-11-01   0.132  -0.680  0.416 -0.132    Normal
2019-12-01   0.235  -1.271 -0.161 -1.197  Moderate
2020-01-01  -0.152  -1.084 -1.032 -2.267  Moderate
2020-02-01  -0.226  -0.062 -1.553 -1.842  Moderate
2020-03-01  -0.192   1.147 -1.581 -0.626    Normal
2020-04-01   0.080   1.967 -1.064  0.983    Normal
2020-05-01   0.081   1.783 -0.620  1.245  Moderate
2020-06-01   0.171   0.731  0.483  1.385  Moderate
2020-07-01 

## Step 12 - Save All Outputs

In [29]:
mdf.to_csv('outputs/full_analysis.csv', index=False)
print('All outputs saved:')
for f in ['monthly_processed.csv','climatology_baseline.csv','period_comparison.csv',
          'future_forecast.csv','future_ces.csv','full_analysis.csv','model_evaluation.csv']:
    print(f'  outputs/{f}')
print('\nPlots saved:')
for f in sorted(os.listdir('plots')):
    print(f'  plots/{f}')


All outputs saved:
  outputs/monthly_processed.csv
  outputs/climatology_baseline.csv
  outputs/period_comparison.csv
  outputs/future_forecast.csv
  outputs/future_ces.csv
  outputs/full_analysis.csv
  outputs/model_evaluation.csv

Plots saved:
  plots/01_rainfall_trend.png
  plots/02_temperature_trend.png
  plots/03_humidity_trend.png
  plots/04_correlation_heatmap.png
  plots/05_distributions.png
  plots/06_anomaly_zscores.png
  plots/07_ces_timeseries.png
  plots/08_ces_distribution.png
  plots/09_anomaly_evolution.png
  plots/10_decade_anomaly.png
  plots/11_seasonal_analysis.png
  plots/12_climate_shift.png
  plots/13_actual_vs_predicted.png
  plots/14_future_forecast.png
  plots/15_future_ces.png


## Step 13 - Automated Climate Insights

In [30]:
print('='*60)
print('BANGALORE CLIMATE INSIGHTS')
print('='*60)

t_early = mdf[mdf['Year'] <= 1980]['Temp_mean'].mean()
t_late  = mdf[mdf['Year'] >= 2000]['Temp_mean'].mean()
print(f'1. Temperature increased by {t_late-t_early:.2f}C from pre-1980 to post-2000.')

a_early = (mdf[mdf['Year'] <= 1980]['any_anom'] == 1).sum()
a_late  = (mdf[mdf['Year'] >= 2000]['any_anom'] == 1).sum()
print(f'2. Anomalies: {a_early} months pre-1980 vs {a_late} months post-2000.')

n_ext = (mdf['CES_label'] == 'Extreme').sum()
pct   = round(n_ext / len(mdf) * 100, 1)
print(f'3. Extreme CES events: {n_ext} months ({pct}% of record).')

mon_rate = mdf[mdf['Season']=='Monsoon']['Rain_label'].apply(lambda x: x!='Normal').mean()*100
print(f'4. Monsoon anomaly rate: {mon_rate:.1f}% of monsoon months.')

if len(period_df) >= 2:
    print(f'5. CES shift: {period_df.iloc[0]["Period"]} to {period_df.iloc[-1]["Period"]}: '
          f'{period_df.iloc[0]["CES"]:.3f} to {period_df.iloc[-1]["CES"]:.3f}')

n_fut_ext = (future_ces['CES_label'] == 'Extreme').sum()
print(f'6. Future: {n_fut_ext}/{FUTURE_STEPS} months projected Extreme.')
print('7. Continued warming and increasing climate variability expected for Bangalore.')
print('='*60)


BANGALORE CLIMATE INSIGHTS
1. Temperature increased by 0.53C from pre-1980 to post-2000.
2. Anomalies: 83 months pre-1980 vs 76 months post-2000.
3. Extreme CES events: 25 months (2.8% of record).
4. Monsoon anomaly rate: 0.0% of monsoon months.
5. CES shift: 1951-1980 to 2001-2024: -0.408 to 0.202
6. Future: 0/24 months projected Extreme.
7. Continued warming and increasing climate variability expected for Bangalore.
